In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# Load the cleaned dataset
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data.csv")

# Rename for compatibility
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2023": "pop_2023",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989"
})

# Convert relevant columns to numeric
for col in [
    'church_persistence_index', 'income_1989_real_2023', 'pct_hs_1990',
    'pop_2023', 'firms_2022', 'firms_1989'
]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop missing or invalid rows
df = df.dropna(subset=[
    'income_1989_real_2023', 'pct_hs_1990',
    'pop_2023', 'firms_2022', 'firms_1989', 'State'
])
df = df[(df['firms_2022'] > 0) & (df['pop_2023'] > 0) & (df['firms_1989'] > 0)]

# Log-transform outcome and controls
df['log_firms_2022'] = np.log(df['firms_2022'])
df['log_pop_2023'] = np.log(df['pop_2023'])
df['log_firms_1989'] = np.log(df['firms_1989'])

# Standardize education and income
scaler = StandardScaler()
df[['income_1989_scaled', 'pct_hs_1990_scaled']] = scaler.fit_transform(
    df[['income_1989_real_2023', 'pct_hs_1990']]
)

# Run the regression (no church persistence, just education + controls)
model_edu_only = smf.ols(
    formula="""
        log_firms_2022 ~ 
        pct_hs_1990_scaled + 
        income_1989_scaled + 
        log_pop_2023 + 
        log_firms_1989 + 
        C(State)
    """,
    data=df
).fit(cov_type='HC3')

# Show the regression results
print(model_edu_only.summary())


                            OLS Regression Results                            
Dep. Variable:         log_firms_2022   R-squared:                       0.963
Model:                            OLS   Adj. R-squared:                  0.963
Method:                 Least Squares   F-statistic:                     1577.
Date:                Thu, 09 Oct 2025   Prob (F-statistic):               0.00
Time:                        11:31:12   Log-Likelihood:                -573.36
No. Observations:                2968   AIC:                             1251.
Df Residuals:                    2916   BIC:                             1562.
Df Model:                          51                                         
Covariance Type:                  HC3                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -4.6468      0